A retriever is a component in langchain that fetches relevant documents from a data source in response to a users' query

- There are multiple types of retrievers
- All retrievers in LangChain are runnables

![Retriever](Retriever.png)

## Wikipedia Retriever

- You give it a query (eg: Albert Einstein)
- It sends query to Wikipedia API
- It retrieves the most relevant articles
- It returns as LangChain Document objects

There is some logic implemented in this, so it is not just a document loader, it is a retriever

In [1]:
from  langchain_community.retrievers import WikipediaRetriever

/tmp/ipykernel_67630/879216472.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from  langchain_community.retrievers import WikipediaRetriever


In [2]:
#initialize the retriever
retriever = WikipediaRetriever(top_k_results=2, lang="en")

In [3]:
query = "the geopolitical history of india and pakistan from the perspective of a chinese"

In [19]:
# Get relevant Wikipedia documents
docs = retriever.invoke(query)

In [ ]:
# print retrieved content
for i, doc in enumerate(docs):
    print(f"\n--- Result {i+1}---")
    print(f"content: \n{doc.page_content}...") 


--- Result 1---
content: 
The India–Pakistan war of 1971, also known as the third Indo-Pakistani war, was a military confrontation between India and Pakistan that occurred during the Bangladesh Liberation War in East Pakistan from 3 December 1971 until the Pakistani capitulation in Dhaka on 16 December 1971.  The war began with Pakistan's Operation Chengiz Khan, consisting of preemptive aerial strikes on eight Indian air stations. The strikes led to India declaring war on Pakistan, marking their entry into the war for East Pakistan's independence, on the side of Bengali nationalist forces. India's entry expanded the existing conflict with Indian and Pakistani forces engaging on both the eastern and western fronts.
Thirteen days after the war started, India achieved a clear upper hand, and the Eastern Command of the Pakistan military signed the instrument of surrender on 16 December 1971 in Dhaka, marking the formation of East Pakistan as the new nation of Bangladesh. Approximately 93,

## Vector Store Retriever (Most common retriever)


In [6]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document
from dotenv import load_dotenv

/home/aromal/anaconda3/envs/llm_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
load_dotenv()


True

In [8]:
documents = [
    Document(page_content="LangChain helps developers build LLM applications easily."),
    Document(page_content="Chroma is a vector database optimized for LLM-based search."),
    Document(page_content="Embeddings convert text into high-dimensional vectors."),
    Document(page_content="OpenAI provides powerful embedding models."),
]

In [9]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [ ]:
vectorstore = Chroma.from_documents(
    documents=documents,
    collection_name="my_collection",
    embedding=embedding_model,
)

In [16]:
retriever = vectorstore.as_retriever(search_kwargs={"k":2})

In [17]:
query = "What is Chroma used for?"
result = retriever.invoke(query)

In [18]:
for i, doc in enumerate(result):
    print(f"\n---Result--- {i+1} ---")
    print(doc.page_content)


---Result--- 1 ---
Chroma is a vector database optimized for LLM-based search.

---Result--- 2 ---
LangChain helps developers build LLM applications easily.


## Maximal Marginal Relevance (MMR)

```How can we pick results that are not only relevant to the query but also different from each other ```

MMR is an information retrieval algorithm designed to reduce redundancy in the retrieved result while maintaining high relevance to the query.

**Why MMR Retriever?**

In regular similarity search, you may get documents that  are:
- All very similar to each other
- Repeating the same info
- Lacking diverse perspective

MMr Retriever avoids that by:
- Picking the most relevant document first
- then picking the next most relevant and least similar to already selected docs
- And so on...

This helps especially in RAG pipeline where:
- You want your context window to contain diverse but still relevant information
- Especially useful when documents are semantically overlapping

In [20]:
# Sample documents
docs = [
    Document(page_content="LangChain makes it easy to work with LLMs."),
    Document(page_content="LangChain is used to build LLM based applications."),
    Document(page_content="Chroma is used to store and search document embeddings."),
    Document(page_content="Embeddings are vector representations of text."),
    Document(page_content="MMR helps you get diverse results when doing similarity search."),
    Document(page_content="LangChain supports Chroma, FAISS, Pinecone, and more."),
]

In [21]:
from langchain_community.vectorstores import FAISS

embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [22]:
vectorstore = FAISS.from_documents(
    documents=docs,
    embedding=embedding_model
)

In [27]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k":3, "lambda_mult":0.5} # lambda_mul = relevance-diversity balance
)

In [28]:
query = "What is langchain?"
result = retriever.invoke(query)

In [29]:
for i , doc in enumerate(result):
    print(f"--- Result --- {i+1} ---")
    print(doc.page_content)

--- Result --- 1 ---
LangChain is used to build LLM based applications.
--- Result --- 2 ---
MMR helps you get diverse results when doing similarity search.
--- Result --- 3 ---
LangChain supports Chroma, FAISS, Pinecone, and more.


## Multi-Query Retriever

```Sometimes a  single query might not capture all the ways information si phrased in your documents ```

For Example:

```How can I stay Healthy```

Could mean:
- What should I eat?
- How often should I exercise?
- How cna I manage stress?

A simple similarity search might miss documents that talk about those things but don't use the word "healthy".


1. Takes your original query
2. Uses a LLM (eg: GPT 3.5) to generate multiple smenatically different versions of that query
3. Performs retrieval for each sub-query
4. Combines and deduplicates the results

In [44]:
from langchain_community.vectorstores import FAISS
from langchain_google_genai import GoogleGenerativeAIEmbeddings, ChatGoogleGenerativeAI
from dotenv import load_dotenv
from langchain_classic.retrievers import MultiQueryRetriever

In [45]:
# Relevant health & wellness documents
all_docs = [
    Document(page_content="Regular walking boosts heart health and can reduce symptoms of depression.", metadata={"source": "H1"}),
    Document(page_content="Consuming leafy greens and fruits helps detox the body and improve longevity.", metadata={"source": "H2"}),
    Document(page_content="Deep sleep is crucial for cellular repair and emotional regulation.", metadata={"source": "H3"}),
    Document(page_content="Mindfulness and controlled breathing lower cortisol and improve mental clarity.", metadata={"source": "H4"}),
    Document(page_content="Drinking sufficient water throughout the day helps maintain metabolism and energy.", metadata={"source": "H5"}),
    Document(page_content="The solar energy system in modern homes helps balance electricity demand.", metadata={"source": "I1"}),
    Document(page_content="Python balances readability with power, making it a popular system design language.", metadata={"source": "I2"}),
    Document(page_content="Photosynthesis enables plants to produce energy by converting sunlight.", metadata={"source": "I3"}),
    Document(page_content="The 2022 FIFA World Cup was held in Qatar and drew global energy and excitement.", metadata={"source": "I4"}),
    Document(page_content="Black holes bend spacetime and store immense gravitational energy.", metadata={"source": "I5"}),
]

In [48]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

In [54]:
model = ChatGoogleGenerativeAI(
    model="models/gemini-2.5-flash-lite"
)

In [50]:
vectorstore = FAISS.from_documents(documents=all_docs, embedding=embedding_model)

In [51]:
similarity_retirever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k":5})

In [55]:
multiquery_retriever = MultiQueryRetriever.from_llm(
    retriever=vectorstore.as_retriever(search_kwargs={"k":5}),
    llm=model
)

In [56]:
query = "How to improve energy levels and maintain balance?"

In [58]:
similarity_result =  similarity_retirever.invoke(query)
multiquery_result = multiquery_retriever.invoke(query)

In [59]:
for i, doc in enumerate(similarity_result):
    print(f"--- Result --- {i+1} ---")
    print(doc.page_content)

--- Result --- 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
--- Result --- 2 ---
The solar energy system in modern homes helps balance electricity demand.
--- Result --- 3 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
--- Result --- 4 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
--- Result --- 5 ---
Regular walking boosts heart health and can reduce symptoms of depression.


In [60]:

for i, doc in enumerate(multiquery_result):
    print(f"--- Result --- {i+1} ---")
    print(doc.page_content)

--- Result --- 1 ---
Drinking sufficient water throughout the day helps maintain metabolism and energy.
--- Result --- 2 ---
Mindfulness and controlled breathing lower cortisol and improve mental clarity.
--- Result --- 3 ---
The solar energy system in modern homes helps balance electricity demand.
--- Result --- 4 ---
Regular walking boosts heart health and can reduce symptoms of depression.
--- Result --- 5 ---
Consuming leafy greens and fruits helps detox the body and improve longevity.
--- Result --- 6 ---
Deep sleep is crucial for cellular repair and emotional regulation.


## Contextual Compression Retriever

The contextual compression Retriever in LangChain is an advanced retriever that improves retrieval quality by compressing documents after retrieval -- keeping only the relevant content based on the user's query's

**Query**:

```What is Photosynthesis```

**Retrieved Document (by a traditional retriever)**
```
The Grand Canyon is a famous natural site.
Photosynthesis is how plants convert light into energy.
Many tourists visit every year
```
This could be caused due the text splitter

**Problem**
- The retriever returns the entire paragraph
- Only one sentence is actually relevant to the query
- The rest is irrelevant noise that wastes context window and may confuse the LLM

**What Contextual Compression Retriever does**:
Returns only the relevant part

**How it works**
1. Base Retriever retrieves N documents
2. A compressor (Usually and LLM) is applied to each document
3. The compressor keeps only the parts relevant to the query
4. Irrelevant content is discarded

**When to use**
- Your documents are long and contain mixed information
- You want to reduce context length of LLM
- You need to improve answer accuracy in RAG pipelines


In [66]:
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_google_genai import GoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from dotenv import load_dotenv
from langchain_classic.retrievers.document_compressors import LLMChainExtractor
from langchain_classic.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_core.documents import Document

In [62]:
# Recreate the document objects from the previous data
docs = [
    Document(page_content=(
        """The Grand Canyon is one of the most visited natural wonders in the world.
        Photosynthesis is the process by which green plants convert sunlight into energy.
        Millions of tourists travel to see it every year. The rocks date back millions of years."""
    ), metadata={"source": "Doc1"}),

    Document(page_content=(
        """In medieval Europe, castles were built primarily for defense.
        The chlorophyll in plant cells captures sunlight during photosynthesis.
        Knights wore armor made of metal. Siege weapons were often used to breach castle walls."""
    ), metadata={"source": "Doc2"}),

    Document(page_content=(
        """Basketball was invented by Dr. James Naismith in the late 19th century.
        It was originally played with a soccer ball and peach baskets. NBA is now a global league."""
    ), metadata={"source": "Doc3"}),

    Document(page_content=(
        """The history of cinema began in the late 1800s. Silent films were the earliest form.
        Thomas Edison was among the pioneers. Photosynthesis does not occur in animal cells.
        Modern filmmaking involves complex CGI and sound design."""
    ), metadata={"source": "Doc4"})
]

In [ ]:
embedding_model = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001"
)
vectorstore = FAISS.from_documents(docs, embedding_model)

In [64]:
base_retriever = vectorstore.as_retriever(search_kwargs={"k":5})

In [65]:
llm = GoogleGenerativeAI(
    model="models/gemini-2.5-flash-lite"
)
compressor = LLMChainExtractor.from_llm(llm)

In [67]:
# Create the contextual compression retriever
compression_retriever = ContextualCompressionRetriever(
    base_retriever=base_retriever,
    base_compressor=compressor
)

In [68]:
query = "What is photosynthesis"
compressed_results = compression_retriever.invoke(query)

In [70]:
for i ,doc in enumerate(compressed_results):
    print(f"--- Result --- {i+1} ---")
    print(doc.page_content)

--- Result --- 1 ---
Photosynthesis is the process by which green plants convert sunlight into energy.
--- Result --- 2 ---
Photosynthesis does not occur in animal cells.
--- Result --- 3 ---
The chlorophyll in plant cells captures sunlight during photosynthesis.
